[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-causal.ipynb)

# Estimating Causal Effects — Weighting, Adjustment and Doubly Robust Estimators

*AIBits Academy · Machine Learning End To End · Advanced · Optional*

The Interpretability page explained why correlation is not causation. This page shows how to estimate a causal effect from observational data when a randomised test is not possible.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

> **⚙ Optional page**
>
> Causal inference has high conceptual value but is a large topic. Treat this page as an introduction, and as the natural next step after the potential-outcomes, randomisation, propensity-matching and causal-diagram sections at the end of the Interpretability page. It is not needed for any project.

## The Question and the Trap

A lender sends WhatsApp reminders before an EMI is due. Did the reminders make customers pay on time? The obvious comparison — on-time rate among customers who got a reminder minus the rate among those who did not — is confounded: *engaged* customers (heavy app users) were both more likely to receive reminders and more likely to pay on time anyway. To know how well any method works we need to know the true answer, so we simulate: 30,000 customers with three covariates (engagement, income, past late payments), a reminder policy that favours engaged customers, and a real effect built in. Because it is a simulation we can compute each customer's outcome both with and without a reminder (the two *potential outcomes*) and therefore the true average treatment effect.

Three standard estimators use the covariates *X*, treatment *T* and outcome *Y*. **Regression adjustment** fits an outcome model μ_t(*x*) = E[*Y* | *T*=*t*, *X*=*x*] and averages μ_1 − μ_0 over everyone. **Inverse-probability weighting (IPW)** fits a propensity model *e*(*x*) = P(*T*=1 | *X*=*x*) and reweights each customer by 1/*e* (if treated) or 1/(1−*e*) (if not), creating a pseudo-population in which treatment is unrelated to *X*. The **doubly robust (augmented IPW) estimator** combines both:

$$\hat\tau_{\text{AIPW}}=\dfrac1n\sum_{i=1}^{n}\left[\hat\mu_1(X_i)-\hat\mu_0(X_i)+\dfrac{T_i\,(Y_i-\hat\mu_1(X_i))}{\hat e(X_i)}-\dfrac{(1-T_i)\,(Y_i-\hat\mu_0(X_i))}{1-\hat e(X_i)}\right]$$

Its remarkable property: the estimate is consistent if *either* the outcome model *or* the propensity model is correct — you get two chances. We test it by deliberately breaking each model in turn (leaving engagement out):

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression

rng = np.random.RandomState(11)
n = 30000
# Do WhatsApp reminders make customers pay their EMI on time?  Engaged customers get more reminders AND pay on time anyway: classic confounding.
engagement = rng.normal(0, 1, n)                                       # app usage score (higher = more engaged)
income = rng.normal(0, 1, n)
past_late = rng.poisson(np.exp(-0.5 - 0.4 * engagement), n)            # number of late payments last year
X = np.column_stack([engagement, income, past_late])

p_treat = 1 / (1 + np.exp(-(-0.2 + 0.9 * engagement + 0.3 * income - 0.35 * past_late)))       # who receives a reminder
T = (rng.rand(n) < p_treat).astype(int)
logit0 = 0.8 + 0.9 * engagement + 0.4 * income - 0.5 * past_late + 0.25 * engagement * income    # on-time payment WITHOUT a reminder
logit1 = logit0 + 0.35                                                 # a reminder adds the same amount on the log-odds scale
p0, p1 = 1 / (1 + np.exp(-logit0)), 1 / (1 + np.exp(-logit1))
Y0, Y1 = (rng.rand(n) < p0).astype(int), (rng.rand(n) < p1).astype(int)   # the two potential outcomes (we can see both only because this is a simulation)
Y = np.where(T == 1, Y1, Y0)
true_ate = (p1 - p0).mean()
print(f"reminded: {T.mean():.1%} of customers.  TRUE average effect of a reminder: {100 * true_ate:+.2f} percentage points")
print(f"naive difference in on-time rates (reminded - not): {100 * (Y[T == 1].mean() - Y[T == 0].mean()):+.2f} points   <- confounded\n")

def fit_outcome(cols, interact=False):
    F = lambda X_, t: np.column_stack([X_[:, cols], t] + ([X_[:, [0]] * X_[:, [1]]] if interact else []))
    m = LogisticRegression(C=1e6, max_iter=5000).fit(F(X, T), Y)
    mu1, mu0 = m.predict_proba(F(X, np.ones(n)))[:, 1], m.predict_proba(F(X, np.zeros(n)))[:, 1]
    return mu1, mu0

def fit_ps(cols):
    return np.clip(LogisticRegression(C=1e6, max_iter=5000).fit(X[:, cols], T).predict_proba(X[:, cols])[:, 1], 0.01, 0.99)

def estimators(ps_cols, out_cols, interact):
    e = fit_ps(ps_cols); mu1, mu0 = fit_outcome(out_cols, interact)
    reg = (mu1 - mu0).mean()                                                                   # regression adjustment (standardisation)
    ipw = (T * Y / e - (1 - T) * Y / (1 - e)).mean()                                           # inverse-probability weighting
    aipw = (mu1 - mu0 + T * (Y - mu1) / e - (1 - T) * (Y - mu0) / (1 - e)).mean()              # doubly robust (augmented IPW)
    return 100 * reg, 100 * ipw, 100 * aipw

print("scenario                                            regression    IPW      doubly robust   (truth %+.2f)" % (100 * true_ate))
for name, ps_cols, out_cols, interact in [("both models correct", [0, 1, 2], [0, 1, 2], True),
                                          ("propensity model omits engagement (wrong)", [1, 2], [0, 1, 2], True),
                                          ("outcome model omits engagement (wrong)", [0, 1, 2], [1, 2], False),
                                          ("BOTH models omit engagement (wrong)", [1, 2], [1, 2], False)]:
    r, i, a = estimators(ps_cols, out_cols, interact)
    print(f"{name:50s}  {r:+7.2f}   {i:+7.2f}   {a:+7.2f}")

| Scenario | Regression | IPW | Doubly robust |
|---|---|---|---|
| Truth | +6.49 | +6.49 | +6.49 |
| Naive difference in means | +24.89 | — | — |
| Both models correct | +6.55 | +6.62 | +6.46 |
| Propensity model omits engagement (wrong) | +6.55 | **+17.88** | +6.47 |
| Outcome model omits engagement (wrong) | **+18.10** | +6.62 | +6.47 |
| Both models wrong | +18.10 | +17.88 | **+17.93** |

The naive comparison claims a reminder adds **24.89** percentage points when the truth is **6.49** — nearly four times too large, all of it confounding. With the right covariates every method recovers the truth (6.46–6.62). Break one model and the single-model estimators fail (regression 18.10 when the outcome model is wrong; IPW 17.88 when the propensity model is wrong) while the doubly robust estimate stays at **6.47**. Break both and nothing helps (17.93): **no estimator can adjust for a confounder that you did not measure or model**.

## Diagnostics You Must Run Before Believing the Number

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

# same simulated reminder data as before
rng = np.random.RandomState(11)
n = 30000
engagement = rng.normal(0, 1, n); income = rng.normal(0, 1, n)
past_late = rng.poisson(np.exp(-0.5 - 0.4 * engagement), n)
X = np.column_stack([engagement, income, past_late]); names = ["engagement", "income", "past_late"]
p_treat = 1 / (1 + np.exp(-(-0.2 + 0.9 * engagement + 0.3 * income - 0.35 * past_late)))
T = (rng.rand(n) < p_treat).astype(int)
logit0 = 0.8 + 0.9 * engagement + 0.4 * income - 0.5 * past_late + 0.25 * engagement * income
p0, p1 = 1 / (1 + np.exp(-logit0)), 1 / (1 + np.exp(-(logit0 + 0.35)))
Y = np.where(T == 1, (rng.rand(n) < p1).astype(int), (rng.rand(n) < p0).astype(int))
e = np.clip(LogisticRegression(C=1e6, max_iter=5000).fit(X, T).predict_proba(X)[:, 1], 0.01, 0.99)

def smd(x, w=None):
    """Standardised mean difference between treated and untreated (|SMD| < 0.1 is the usual 'balanced' rule of thumb)."""
    w = np.ones(len(x)) if w is None else w
    m1 = np.average(x[T == 1], weights=w[T == 1]); m0 = np.average(x[T == 0], weights=w[T == 0])
    v1 = np.average((x[T == 1] - m1) ** 2, weights=w[T == 1]); v0 = np.average((x[T == 0] - m0) ** 2, weights=w[T == 0])
    return (m1 - m0) / np.sqrt((v1 + v0) / 2)

weights = np.where(T == 1, 1 / e, 1 / (1 - e))
print("covariate balance: standardised mean difference, reminded vs not reminded")
print("  covariate       before weighting    after inverse-probability weighting")
for j, nm in enumerate(names):
    print(f"  {nm:12s}       {smd(X[:, j]):+.3f}                 {smd(X[:, j], weights):+.3f}")

print(f"\noverlap: propensity scores range from {e.min():.3f} to {e.max():.3f}; reminded customers: median {np.median(e[T == 1]):.2f}, not reminded: median {np.median(e[T == 0]):.2f}")
lo, hi = np.quantile(e, [0.01, 0.99])
print(f"share of customers with a propensity below 0.05 or above 0.95 (near-deterministic treatment): {np.mean((e < 0.05) | (e > 0.95)):.1%}")
print(f"largest inverse-probability weight: {weights.max():.1f}; effective sample size after weighting: {weights.sum() ** 2 / (weights ** 2).sum():,.0f} of {n:,}")

# sensitivity to a hidden confounder: the E-value (VanderWeele & Ding). How strong would an unmeasured confounder have to be to explain the effect away?
p1_hat = np.average(Y[T == 1], weights=weights[T == 1]); p0_hat = np.average(Y[T == 0], weights=weights[T == 0])
rr = p1_hat / p0_hat
e_value = rr + np.sqrt(rr * (rr - 1))
print(f"\nadjusted risk ratio (on-time rate with vs without reminder): {rr:.3f};  E-value = {e_value:.2f}")
print(f"an unmeasured factor would need to raise BOTH the chance of getting a reminder AND of paying on time by a factor of about {e_value:.2f} to explain the estimate away")

| Check | What we found | Reading |
|---|---|---|
| Covariate balance (standardised mean difference) | engagement +0.866 → +0.017; income +0.238 → +0.010; past_late −0.464 → −0.010 | Weighting balances every measured covariate (target \|SMD\| &lt; 0.1) |
| Overlap (positivity) | propensity range 0.010–0.975; only 1.4% of customers below 0.05 or above 0.95 | Every kind of customer has a real chance of either treatment: good |
| Weights | largest weight 31.7; effective sample size 20,869 of 30,000 | A few customers count a lot; the weighting costs a third of the sample's information |
| Sensitivity to a hidden confounder (E-value) | risk ratio 1.129; E-value 1.51 | An unmeasured factor associated with both treatment and outcome by a factor of about 1.5 could explain the effect away |

**Balance** is the diagnostic that matters most: before weighting, reminded customers were 0.87 standard deviations more engaged than the others; after, the difference is 0.017. **Overlap** asks whether comparable customers exist in both groups; where the propensity is close to 0 or 1 there is nobody to compare with, the weights explode, and the estimate rests on extrapolation — trim or restrict the population rather than trust it. The **E-value** is a way of saying how fragile a result is: a small E-value (here 1.51, roughly “a moderately strong confounder”) tells you to be cautious and to look for unmeasured drivers; a large one means only a very strong hidden factor could overturn the finding. It is a communication tool, not a proof.

## The Assumptions Behind the Numbers

- **No unmeasured confounding (ignorability):** once you condition on the covariates, treatment is as good as random. Untestable; argued from domain knowledge and probed with sensitivity analysis.
- **Positivity (overlap):** every covariate profile has a non-zero chance of each treatment. Checkable, as above.
- **Consistency and no interference:** the treatment is well defined, and one customer’s treatment does not change another’s outcome (violated by, say, referral incentives).
- **Covariates measured before treatment.** Adjusting for something the treatment caused (a mediator or a collider) can create bias instead of removing it — see the causal diagrams on the Interpretability page.

> **⚠ An experiment beats any estimator**
>
> Everything above is a second-best. If you can randomise, do (see the A/B testing project): randomisation removes confounding by design, including confounders you never thought of. Use observational estimators when an experiment is impossible, unethical or already in the past, report the assumptions with the number, and prefer “the estimated effect, assuming no unmeasured confounding, is…” to “the reminder causes…”.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Setup for the exercises

The cell below simulates the reminder data used on the page.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

def simulate(seed=11, n=30000):
    """WhatsApp-reminder data: engaged customers get more reminders AND pay on time anyway (confounding). True effect: +0.35 on the log-odds scale."""
    rng = np.random.RandomState(seed)
    engagement = rng.normal(0, 1, n); income = rng.normal(0, 1, n)
    past_late = rng.poisson(np.exp(-0.5 - 0.4 * engagement), n)
    X = np.column_stack([engagement, income, past_late])
    T = (rng.rand(n) < 1 / (1 + np.exp(-(-0.2 + 0.9 * engagement + 0.3 * income - 0.35 * past_late)))).astype(int)
    logit0 = 0.8 + 0.9 * engagement + 0.4 * income - 0.5 * past_late + 0.25 * engagement * income
    p0, p1 = 1 / (1 + np.exp(-logit0)), 1 / (1 + np.exp(-(logit0 + 0.35)))
    Y0, Y1 = (rng.rand(n) < p0).astype(int), (rng.rand(n) < p1).astype(int)
    return X, T, np.where(T == 1, Y1, Y0), (p1 - p0).mean()

X, T, Y, true_ate = simulate()
ps = np.clip(LogisticRegression(C=1e6, max_iter=5000).fit(X, T).predict_proba(X)[:, 1], 0.01, 0.99)    # estimated propensity scores
print(f"reminded: {T.mean():.1%}; true average effect: {true_ate:+.4f}")

### Exercise 1 · Easy · The confounded comparison

Run the setup cell above (it simulates 30,000 customers and prints the true effect). Compute `naive`, the difference in on-time rate between reminded and not-reminded customers, as a fraction (not percentage points).

In [ ]:
naive = None   # TODO


In [ ]:
try:
    check("naive difference is about +0.249", abs(naive - 0.2489) < 0.003)
    check("it is much larger than the truth", naive > 3 * true_ate)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
naive = float(Y[T == 1].mean() - Y[T == 0].mean())
```

</details>


### Exercise 2 · Medium · Inverse-probability weighting

Using the estimated propensity scores `ps`, compute the IPW estimate `ipw` of the average effect: the mean of `T*Y/ps - (1-T)*Y/(1-ps)`.

In [ ]:
ipw = None   # TODO


In [ ]:
try:
    check("IPW is close to the truth", abs(ipw - true_ate) < 0.01)
    check("and far from the naive difference", abs(ipw - 0.2489) > 0.15)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
ipw = float((T * Y / ps - (1 - T) * Y / (1 - ps)).mean())
```

</details>


### Exercise 3 · Stretch · Did weighting balance the groups?

Compute the standardised mean difference (SMD) of `engagement` (column 0 of `X`) between reminded and not-reminded customers, first unweighted (`smd_before`) and then with inverse-probability weights `w = T/ps + (1-T)/(1-ps)` (`smd_after`). SMD = (mean1 - mean0) / sqrt((var1 + var0)/2), using weighted means and variances for the second.

In [ ]:
import numpy as np
x = X[:, 0]
w = T / ps + (1 - T) / (1 - ps)
smd_before = smd_after = None   # TODO


In [ ]:
try:
    check("before weighting the groups differ a lot (> 0.8)", abs(smd_before) > 0.8)
    check("after weighting they are balanced (< 0.05)", abs(smd_after) < 0.05)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
x = X[:, 0]
w = T / ps + (1 - T) / (1 - ps)
def smd(x, w):
    m1 = np.average(x[T == 1], weights=w[T == 1]); m0 = np.average(x[T == 0], weights=w[T == 0])
    v1 = np.average((x[T == 1] - m1) ** 2, weights=w[T == 1]); v0 = np.average((x[T == 0] - m0) ** 2, weights=w[T == 0])
    return (m1 - m0) / np.sqrt((v1 + v0) / 2)
smd_before = smd(x, np.ones(len(x))); smd_after = smd(x, w)
```

</details>


---
*Back to the course: **Machine Learning End To End → Estimating Causal Effects — Weighting, Adjustment and Doubly Robust Estimators**.*